In [1]:
import pandas as pd
import table_convert
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage
from tqdm.notebook import tqdm as log_progress
import os
import re
import time
import cga_utils
from langchain_community.chat_models import ChatOllama

In [7]:
from langchain.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path=".langchain.ollama38.db"))

llm = ChatOllama(model="qwen3:4b", temperature = 0.0, top_p = 1, repeat_penalty=1, presence_penalty=0, frequency_penalty=0) 



In [14]:
v18r_errors = pd.read_csv("res/e38_18r.csv").query('exact_match == False')
error_summaries=pd.read_csv('res/error_summary_e38_v18r.csv')
v18r_errors["error_summary"]=error_summaries["problem"]


In [57]:

messages = [("human","""
The following list contains the result of a QA prediction task that involves calculation. 
The calculation is done by a Python code, generated by an LLM.
The list items are dict, containing a question, and a description of the wrong behavior of the code.    
Your task is to find patterns in the questions, and group them together if they tend to fail in a similar way. You can use the error description as well. 
The output format is  a list of tuples, where a tuple has the form (question_index, group_tag)
The LIST is:
{q_list}
/no_think"""
)]

prompt = ChatPromptTemplate.from_messages(messages)
    
output_parser = StrOutputParser()

chain = prompt | llm #| output_parser
target_errors = v18r_errors.query("calc_pattern=='(#-#)/#'")

q_list=[(item["question"], item["error_summary"]) for i,item in target_errors.iterrows()]
response = chain.invoke({"q_list":  q_list})



In [58]:
response

AIMessage(content="<think>\n\n</think>\n\n[('0', 'percentage_change_calculation'), ('1', 'scale_mismatch'), ('2', 'missing_scale'), ('3', 'scale_based'), ('4', 'percentage_change_calculation'), ('5', 'scale_mismatch'), ('6', 'scale_mismatch'), ('7', 'data_extraction'), ('8', 'scale_mismatch'), ('9', 'scale_mismatch'), ('10', 'percentage_change_calculation'), ('11', 'percentage_change_calculation'), ('12', 'scale_calculation'), ('13', 'nan'), ('14', 'nan'), ('15', 'nan'), ('16', 'nan'), ('17', 'nan'), ('18', 'nan'), ('19', 'nan'), ('20', 'nan'), ('21', 'nan'), ('22', 'nan'), ('23', 'nan'), ('24', 'nan'), ('25', 'nan'), ('26', 'nan'), ('27', 'nan'), ('28', 'nan'), ('29', 'nan'), ('30', 'nan'), ('31', 'nan'), ('32', 'nan'), ('33', 'nan'), ('34', 'nan'), ('35', 'nan'), ('36', 'nan'), ('37', 'nan'), ('38', 'nan'), ('39', 'nan'), ('40', 'nan'), ('41', 'nan'), ('42', 'nan'), ('43', 'nan'), ('44', 'nan'), ('45', 'nan'), ('46', 'nan'), ('47', 'nan'), ('48', 'nan'), ('49', 'nan'), ('50', 'nan'),

In [39]:
messages = [("human","""
The following list contains questions, that are part of a QA dataset.
Your task is to find financial terms in the questions, that describe the type of calculation.   
The output format is a list of the found financial terms.
The LIST is:
{q_list}
/no_think"""
)]
#/no_think
prompt = ChatPromptTemplate.from_messages(messages)
    
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
target_errors = v18r_errors.query("calc_pattern=='(#-#)/#'")

q_list=list(target_errors["question"])[:-1]
response = chain.invoke({"q_list":  q_list})
response

'<think>\n\n</think>\n\n["percentage change", "percentage change", "percentage change", "percentage change", "percentage increase", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "net percentage difference", "percentage change", "percentage change", "change in total cost of revenue", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "percentage change", "perce

In [29]:
len(target_errors)

39

In [53]:
errors = pd.read_csv("res/e38_18r.csv").query('exact_match == False')
messages = [("human","""
You received a question. 
Your task is to find a financial term in the questions that describes the type of calculation.   
The output format is the found financial term (max 5 words).
The Question is:
{question}
/no_think"""
)]
#/no_think

target_errors = v18r_errors.query("calc_pattern=='(#-#)/#'")
res = []
for i, item in log_progress(errors.iterrows()):   
    if not item["exact_match"]:
        prompt = ChatPromptTemplate.from_messages(messages)
    
        output_parser = StrOutputParser()

        chain = prompt | llm | output_parser
        response = chain.invoke({"question":  item["question"]})
        response = response.replace("<think>\n\n</think>\n\n", "")
        print(item["calc_pattern"], response)
        res.append({"qid": item["qid"], "question": item["question"], "term": response})
pd.DataFrame(res).to_csv('res/terms_e38_v18r.csv') 

0it [00:00, ?it/s]

(#+#)/# Average defined contribution schemes
(#+#)/# Average defined benefit schemes
[(#+#)/#]-[(#+#)/#] Average defined contribution schemes
(#+#)/# Average Free Cash Flow
(#+#)/# Average Free Cash Flow
[(#+#)/#]-[(#+#)/#] Change in average free cash flow
((#-#)+(#-#))/# Average Difference
#-# Total expenses
#+#+# Total Assets
#+# Liabilities and Stockholders' Equity
#-# Difference in amount between Deferred Revenue and Other non-current liabilities
(#-#)/# Percentage change
(#+#+#)/# Average tax rate
-#-# Change in preferred stock disposition
#-# Change in closing cash
(#+#)/# Average Total Current Tax Expense
(#+#)/# Average Total Current Tax Expense
[(#+#)/#]-[(#+#)/#] Change in average total current tax expense
#+# Count of nonvested shares
-(#+#+#)/# Average Tax Exempt Interest Income
-#/#-# Percentage change
(#%+#%)/# Average Dividend Yield
(#-#)/# Percentage change
(#*#)+(#*#) Share-based compensation expense
#/# Share issuance ratio
#*# Market capitalization
(#-#)/# Percentage

In [51]:
pd.DataFrame(res)["term"].value_counts()

term
Percentage change                       40
Percentage increase                      3
Average defined contribution schemes     2
Nominal difference                       2
Share-based compensation expense         2
                                        ..
Change in Total Gross Margin             1
liability to asset ratio                 1
Equity acquisition value                 1
Percentage Difference                    1
Forfeiture rate                          1
Name: count, Length: 149, dtype: int64